# Brain tumor MRI Classification

## Importing libaries

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
import os

## loading the training and testing dataset

In [ ]:
train_dir = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training'
test_dir = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing'

## loading data with argumnetor

Here the argumentor rotating/shifting/zooming slightly creates "new" variations of existing images, which helps the model generalize instead of memorizing. We only augment the training set — validation/test should reflect real, unaltered data. class_mode='categorical' because we have 4 classes with one-hot labels. shuffle=False on test data matters later — we need predictions to line up in order with true labels for the confusion matrix. here a fix has been done in the ImageDataGenerator function as the original ualues are retained here. so the images are used by the trained model in the pixels which is needed.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

from tensorflow.keras.applications.efficientnet import preprocess_input

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,   # replaces rescale=1./255
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.15
)

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

test_gen = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False   # important for later evaluation/confusion matrix
)

print(train_gen.class_indices)  # confirm class → label mapping

## Visualizing the loaded dataset

In [ ]:
images, labels = next(train_gen)
class_names = list(train_gen.class_indices.keys())

plt.figure(figsize=(10,10))
for i in range(9):
    plt.subplot(3,3,i+1)
    plt.imshow(images[i])
    plt.title(class_names[np.argmax(labels[i])])
    plt.axis('off')
plt.show()

## enchanced visualization
for better underatnding of the dataset,
Caution: This is one of the most common real-world bugs with pretrained models: every pretrained backbone has its own expected input format, and mismatching it silently breaks training without throwing an error — the code runs fine, it just doesn't learn.

In [ ]:
images, labels = next(train_gen)
class_names = list(train_gen.class_indices.keys())

plt.figure(figsize=(10,10))
for i in range(9):
    plt.subplot(3,3,i+1)
    plt.imshow(images[i] / 255.0)   # normalize just for display
    plt.title(class_names[np.argmax(labels[i])])
    plt.axis('off')
plt.show()

## Building the model

Here we use the pattern of transfer learning. why do we use the following things?
Why this shape:

include_top=False strips EfficientNet's original classification layer (built for 1000 ImageNet classes) — we're keeping just the "feature extractor" part that knows how to see edges, textures, shapes.

base_model.trainable = False freezes those pretrained weights initially. We train only the new layers first so we don't wreck good pretrained features with random noise from an untrained head.

GlobalAveragePooling2D collapses the feature maps into a single vector per image — much fewer parameters than Flatten, less overfitting risk.
Dropout randomly disables neurons during training to prevent overfitting — important since our dataset is small.
Final Dense(4, softmax) gives probability across our 4 tumor classes.

categorical_crossentropy is the standard loss for multi-class classification with one-hot labels.

In [ ]:
base_model = EfficientNetB0(
    include_top=False,          # drop the original 1000-class ImageNet head
    weights='imagenet',         # load pretrained weights
    input_shape=(224, 224, 3)
)

base_model.trainable = False    # freeze it for now — don't destroy pretrained features yet

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(4, activation='softmax')(x)   # 4 classes

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10
)

## Model Evaluation 

90.8% train accuracy and 90.8% validation accuracy, close together, which means the model is generalizing well and not overfitting.

## fine tuning of the model

the base model was completely frozen — only your new classification head learned. Now we unfreeze the last 20 layers of EfficientNet so it can slightly adapt its high-level features (which were tuned for ImageNet objects) toward MRI-specific patterns like tumor textures and shapes. We use a much smaller learning rate (1e-5 instead of 1e-3) so we nudge these pretrained weights gently rather than overwriting the useful knowledge they already have — too high a learning rate here would wreck the pretrained features (this is called "catastrophic forgetting")

In [ ]:
base_model.trainable = True

# freeze all but the last ~20 layers of EfficientNet
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # much smaller LR
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10
)

## Evaluating the test data

Validation data was still used indirectly during training (to monitor and could influence when you'd stop). The test set is completely held out — this is the number that actually represents real-world performance, and it's the one you should quote in your README, not validation accuracy.

classification_report gives you precision, recall, and F1-score per class — important because a model can have great overall accuracy while being terrible at one specific class (e.g., mixing up glioma and meningioma, which look similar)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# get true labels and predictions
test_gen.reset()
y_true = test_gen.classes
y_pred_probs = model.predict(test_gen)
y_pred = np.argmax(y_pred_probs, axis=1)

class_names = list(test_gen.class_indices.keys())

# classification report
print(classification_report(y_true, y_pred, target_names=class_names))

## Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Brain Tumor Classification')
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Saving the model

In [ ]:
model.save('brain_tumor_model.keras')

In [ ]:
def get_gradcam_heatmap(model, img_array, last_conv_layer_name, pred_index=None):
    base = model.get_layer('efficientnetb0')
    
    # model that outputs the conv layer's activations
    conv_model = tf.keras.Model(base.inputs, base.get_layer(last_conv_layer_name).output)
    
    with tf.GradientTape() as tape:
        inputs = tf.cast(img_array, tf.float32)
        conv_outputs = conv_model(inputs)
        tape.watch(conv_outputs)   # we need gradients w.r.t. this intermediate tensor
        
        # manually pass conv_outputs through the REMAINING layers of the full model
        x = conv_outputs
        for layer in model.layers[2:]:   # skip input_layer and efficientnetb0
            x = layer(x)
        predictions = x
        
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    
    return heatmap.numpy(), pred_index.numpy() if pred_index is None else pred_index

In [ ]:
# find the name of EfficientNet's last conv layer
for layer in base_model.layers[::-1]:
    if 'conv' in layer.name.lower():
        print(layer.name)
        break

Loads the image fresh and preprocesses it the same way training data was (must match, or the model sees garbage).
Gets the model's prediction and confidence for that image.
Calls the Grad-CAM function from Step 11 to get the raw heatmap.
Resizes the heatmap (which comes out small, e.g. 7×7, since it's from a deep conv layer) up to 224×224 to match the original image.
Applies a color map (JET — blue=low importance, red=high importance) and blends it on top of the original image at 40% opacity (alpha=0.4) so you can see both the brain scan and the heatmap together.
Saves the figure — these are your README images.

In [ ]:
def display_gradcam(img_path, model, last_conv_layer_name, class_names, alpha=0.4):
    # load and preprocess image
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=(224, 224))
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array_batch = np.expand_dims(img_array, axis=0)
    img_array_preprocessed = preprocess_input(img_array_batch.copy())
    
    # get prediction
    preds = model.predict(img_array_preprocessed)
    pred_index = np.argmax(preds[0])
    confidence = preds[0][pred_index]
    
    # get heatmap
    heatmap, _ = get_gradcam_heatmap(model, img_array_preprocessed, last_conv_layer_name, pred_index)
    
    # resize heatmap to match original image
    heatmap_resized = cv2.resize(heatmap, (224, 224))
    heatmap_colored = np.uint8(255 * heatmap_resized)
    heatmap_colored = cv2.applyColorMap(heatmap_colored, cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    
    # overlay heatmap on original image
    superimposed = heatmap_colored * alpha + img_array
    superimposed = np.clip(superimposed, 0, 255).astype('uint8')
    
    # plot side by side
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_array.astype('uint8'))
    axes[0].set_title('Original MRI')
    axes[0].axis('off')
    
    axes[1].imshow(heatmap_resized, cmap='jet')
    axes[1].set_title('Grad-CAM Heatmap')
    axes[1].axis('off')
    
    axes[2].imshow(superimposed)
    axes[2].set_title(f'Overlay\nPred: {class_names[pred_index]} ({confidence:.2%})')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.savefig(f'gradcam_{class_names[pred_index]}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
import random

test_classes = os.listdir(test_dir)
for cls in test_classes:
    cls_folder = os.path.join(test_dir, cls)
    sample_img = random.choice(os.listdir(cls_folder))
    img_path = os.path.join(cls_folder, sample_img)
    display_gradcam(img_path, model, 'top_conv', class_names)

In [ ]:
import json

# 1. Save the model (skip if already done)
model.save('brain_tumor_model.keras')

# 2. Save classification report as text file
report = classification_report(y_true, y_pred, target_names=class_names)
with open('classification_report.txt', 'w') as f:
    f.write(report)

# 3. Save class_indices mapping (you'll need this in Streamlit later)
with open('class_indices.json', 'w') as f:
    json.dump(train_gen.class_indices, f)

# 4. Confirm everything's in the working directory
import os
print(os.listdir('.'))